# Avida CFO – contract extraction (runs on OpenShift AI Workbench)

This notebook builds the knowledge base the **Contract Agent** uses:

1. Render the demo contracts (`backend/data/contracts/*.md`) as PDFs (in real life the owner uploads PDFs).
2. Extract text page by page with `pypdf`.
3. Ask the **vLLM-served model on OpenShift AI** to extract key terms + clauses as structured JSON.
4. Benchmark the extraction against the hand-labelled ground truth.
5. Write `backend/data/contracts_index.json`.

Run it in a JupyterLab workbench in the same cluster (the internal service URL works there).

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, os
from pathlib import Path
from openai import OpenAI

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = REPO / "backend" / "data"
CONTRACTS = DATA / "contracts"

LLM_BASE_URL = os.getenv("LLM_BASE_URL", "http://granite-8b-predictor.avida-models.svc.cluster.local:8080/v1")
LLM_MODEL = os.getenv("LLM_MODEL", "granite-8b")
client = OpenAI(base_url=LLM_BASE_URL, api_key=os.getenv("LLM_API_KEY", "EMPTY"))
print([m.id for m in client.models.list().data])

['granite-8b']


## 1. Render the demo contracts as PDF

In [3]:
docs = {}
for md_file in sorted(CONTRACTS.glob("*.md")):
    text = md_file.read_text(encoding="utf-8")
    pages = ["## " + p if i else p for i, p in enumerate(text.split("\n## "))]
    docs[md_file.name] = pages
{k: len(v) for k, v in docs.items()}

{'cloudhost_msa.md': 7, 'paperpro_supply.md': 5}

## 2. Extract text per page

## 3. LLM extraction with a JSON schema (vLLM guided decoding)

In [4]:
SCHEMA = {
    "type": "object",
    "properties": {
        "supplier": {"type": "string"},
        "title": {"type": "string"},
        "start_date": {"type": "string"},
        "monthly_fee": {"type": "number"},
        "fixed_price_until": {"type": ["string", "null"]},
        "price_change_notice_days": {"type": ["integer", "null"]},
        "max_annual_increase_pct": {"type": ["number", "null"]},
        "clauses": {"type": "array", "items": {"type": "object", "properties": {
            "id": {"type": "string"}, "page": {"type": "integer"}, "title": {"type": "string"}, "text": {"type": "string"}},
            "required": ["id", "page", "title", "text"]}},
    },
    "required": ["supplier", "title", "start_date", "monthly_fee", "fixed_price_until",
                 "price_change_notice_days", "max_annual_increase_pct", "clauses"],
}
SYSTEM = (
    "You extract key commercial terms from a supplier contract for a small company. "
    "Dates as YYYY-MM-DD. fixed_price_until = last day of any price guarantee, else null. "
    "Notice periods in days (3 months = 90). Only include clauses about fees, prices, indexation, notice or term; "
    "copy their text verbatim and give the page number. The supplier is the party that is not the Customer. JSON only."
)

def extract(pages: list[str]) -> dict:
    text = "\n\n".join(f"[page {i+1}]\n{t}" for i, t in enumerate(pages))
    r = client.chat.completions.create(
        model=LLM_MODEL, temperature=0,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}],
        response_format={"type": "json_schema", "json_schema": {"name": "contract", "schema": SCHEMA}},
    )
    return json.loads(r.choices[0].message.content)

extracted = {name: extract(pages) for name, pages in docs.items()}
extracted

{'cloudhost_msa.md': {'supplier': 'CloudHost GmbH',
  'title': 'Master Services Agreement – Business ERP Hosting',
  'start_date': '2025-09-01',
  'monthly_fee': 9800,
  'fixed_price_until': '2027-08-15',
  'price_change_notice_days': 90,
  'max_annual_increase_pct': None,
  'clauses': [{'id': '§5.1',
    'page': 6,
    'title': 'Fees',
    'text': 'The Customer pays a monthly fee of EUR 9,800.00 (net) for the Business ERP Hosting plan, invoiced on the 15th of each month.'},
   {'id': '§5.2',
    'page': 6,
    'title': 'Price guarantee',
    'text': 'The fees under §5.1 are fixed for a period of 24 months from the commencement date (1 September 2025). Price increases during this period are excluded.'},
   {'id': '§5.3',
    'page': 6,
    'title': 'Price adjustments after the fixed period',
    'text': "After expiry of the price guarantee, the Provider may adjust the fees with at least three (3) months' written notice. The Customer may terminate the agreement with effect from the date

## 4. Benchmark against the hand-labelled index

In [5]:
truth = {c["supplier"]: c for c in json.loads((DATA / "contracts_index.json").read_text(encoding="utf-8"))["contracts"]}
FIELDS = ["start_date", "monthly_fee", "fixed_price_until", "price_change_notice_days", "max_annual_increase_pct"]

def match_truth(supplier: str) -> dict:
    return next((v for k, v in truth.items() if k.split()[0].lower() in supplier.lower()), {})

rows, hits = [], 0
for name, ex in extracted.items():
    gt = match_truth(ex["supplier"])
    if not gt:
        print("no ground truth for", ex["supplier"]); continue
    for f in FIELDS:
        ok = ex.get(f) == gt.get(f)
        hits += ok
        rows.append((gt["supplier"], f, gt.get(f), ex.get(f), "OK" if ok else "MISMATCH"))
for r in rows:
    print(*r, sep=" | ")
print(f"\nField accuracy: {hits}/{len(rows)} = {hits/len(rows):.0%}")

CloudHost GmbH | start_date | 2025-09-01 | 2025-09-01 | OK
CloudHost GmbH | monthly_fee | 9800 | 9800 | OK
CloudHost GmbH | fixed_price_until | 2027-08-31 | 2027-08-15 | MISMATCH
CloudHost GmbH | price_change_notice_days | 90 | 90 | OK
CloudHost GmbH | max_annual_increase_pct | None | None | OK
PaperPro GmbH | start_date | 2024-01-01 | 2024-01-01 | OK
PaperPro GmbH | monthly_fee | 1200 | 1200.0 | OK
PaperPro GmbH | fixed_price_until | None | None | OK
PaperPro GmbH | price_change_notice_days | 30 | 30 | OK
PaperPro GmbH | max_annual_increase_pct | 5.0 | 5 | OK

Field accuracy: 9/10 = 90%


## 5. Write the index used by the backend

Keeps ids/aliases from the existing file. Set `OVERWRITE = True` only if the benchmark looks good.

In [6]:
OVERWRITE = False

out = []
for i, (name, ex) in enumerate(sorted(extracted.items()), start=1):
    gt = match_truth(ex["supplier"])
    out.append({
        "id": gt.get("id", f"C-{i:03d}"),
        "aliases": gt.get("aliases", [ex["supplier"].split()[0].lower()]),
        "file": f"contracts/{name}",
        **ex,
    })

target = DATA / ("contracts_index.json" if OVERWRITE else "contracts_index.generated.json")
target.write_text(json.dumps({"_comment": "Generated by notebooks/contract_extraction.ipynb", "contracts": out},
                             indent=2, ensure_ascii=False), encoding="utf-8")
print("written", target)

written /opt/app-root/src/avida-cfo/backend/data/contracts_index.generated.json
